# Phase 6: Execution, Market Microstructure & Systems  
## Notebook 06_07 — Limit Board, Margin, and Deadband Rebalancing

### Research question

How do exchange price-limit boards, futures margin requirements, contract rounding, unreachable trades, and deadband rebalancing change a strategy’s actual portfolio path?

This notebook follows:

```text
06_01_almgren_chriss_optimal_execution
06_02_dynamic_programming_execution
06_03_square_root_market_impact_model
06_04_hawkes_process_order_flow
06_05_rough_volatility_estimation
06_06_microstructure_friction_cpp_core
06_07_limit_board_margin_and_deadband_rebalancing
```

The previous notebook built a microstructure friction core. This notebook moves into a futures-style portfolio/execution constraint layer.

It covers:

1. futures contract metadata;
2. tick size and contract multiplier;
3. initial and maintenance margin;
4. daily price-limit bands;
5. limit-up and limit-down boards;
6. locked markets and unreachable trades;
7. contract rounding;
8. target notional and target contracts;
9. deadband rebalancing;
10. no-trade zones;
11. margin utilisation;
12. forced deleveraging;
13. cash and variation margin;
14. turnover reduction from deadbands;
15. tracking error from deadbands;
16. limit-board missed-trade attribution;
17. stress scenarios;
18. governance flags;
19. audit checks;
20. saved outputs and manifest.

The key idea:

> In futures trading, the desired target weight is only the start. Price limits, margin, contract granularity, and deadbands decide what portfolio you can actually hold.

## 1. Why this matters

A clean backtest often assumes:

$$
target\ weight \rightarrow executed\ weight
$$

But futures markets add constraints:

- contracts are discrete;
- prices move in ticks;
- exchanges impose daily price limits;
- markets can lock limit-up or limit-down;
- margin requirements change effective leverage;
- cash must absorb variation margin;
- rebalancing too often creates unnecessary turnover;
- rebalancing too rarely creates tracking error.

The realistic transition is:

```text
target weight
→ target notional
→ target contracts
→ rounded contracts
→ price-limit check
→ margin check
→ deadband check
→ executable trade
→ actual holdings
```

## 2. Limit-board intuition

For a futures contract with previous settlement price $S_{t-1}$, a daily limit percentage $L$ creates:

$$
limit\_up_t = S_{t-1}(1+L)
$$

$$
limit\_down_t = S_{t-1}(1-L)
$$

If the market is limit-up, aggressive buy orders may be impossible or extremely unlikely to fill.

If the market is limit-down, aggressive sell orders may be impossible or extremely unlikely to fill.

This notebook uses a simplified rule:

| State | Blocked trade |
|---|---|
| limit up | buy trade blocked |
| limit down | sell trade blocked |

This is deliberately conservative.

## 3. Deadband rebalancing

Deadband rebalancing avoids tiny trades.

If current weight is $w_t$ and target weight is $w_t^*$, trade only if:

$$
|w_t^*-w_t| > band
$$

Otherwise:

$$
trade = 0
$$

A deadband reduces turnover and costs, but increases tracking error.

This notebook compares:

1. no deadband;
2. small deadband;
3. medium deadband;
4. large deadband;
5. volatility-scaled deadband.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class LimitBoardMarginConfig:
    n_days: int = 1_200
    seed: int = 42
    annualisation: int = 252
    initial_equity: float = 10_000_000.0
    gross_target: float = 1.20
    max_abs_weight: float = 0.30
    base_deadband: float = 0.015
    medium_deadband: float = 0.030
    large_deadband: float = 0.050
    rebalance_frequency_days: int = 1
    commission_per_contract: float = 2.0
    slippage_ticks: float = 0.5
    max_margin_utilisation: float = 0.65
    maintenance_margin_buffer: float = 1.10
    forced_deleverage_fraction: float = 0.25
    limit_event_probability: float = 0.012
    stress_limit_event_probability: float = 0.050
    output_subdir: str = "limit_board_margin_deadband_rebalancing_v1"

config = LimitBoardMarginConfig()
config

## 5. Futures contract metadata

We simulate a small futures universe with different:

- tick sizes;
- contract multipliers;
- margin rates;
- daily limit percentages;
- volatility assumptions;
- liquidity levels.

The metadata layer is critical. Without it, futures backtests quietly become wrong.

In [ ]:
def make_contract_metadata():
    metadata = pd.DataFrame([
        {
            "contract": "EQ_IDX",
            "asset_class": "equity_index",
            "initial_price": 4500.0,
            "tick_size": 0.25,
            "contract_multiplier": 50.0,
            "initial_margin_rate": 0.09,
            "maintenance_margin_rate": 0.075,
            "daily_limit_pct": 0.07,
            "ann_vol": 0.18,
            "adv_contracts": 350_000,
        },
        {
            "contract": "BOND_10Y",
            "asset_class": "rates",
            "initial_price": 112.0,
            "tick_size": 1.0 / 64.0,
            "contract_multiplier": 1000.0,
            "initial_margin_rate": 0.035,
            "maintenance_margin_rate": 0.030,
            "daily_limit_pct": 0.035,
            "ann_vol": 0.08,
            "adv_contracts": 220_000,
        },
        {
            "contract": "CRUDE",
            "asset_class": "energy",
            "initial_price": 75.0,
            "tick_size": 0.01,
            "contract_multiplier": 1000.0,
            "initial_margin_rate": 0.12,
            "maintenance_margin_rate": 0.10,
            "daily_limit_pct": 0.10,
            "ann_vol": 0.35,
            "adv_contracts": 450_000,
        },
        {
            "contract": "COPPER",
            "asset_class": "metals",
            "initial_price": 4.0,
            "tick_size": 0.0005,
            "contract_multiplier": 25_000.0,
            "initial_margin_rate": 0.11,
            "maintenance_margin_rate": 0.09,
            "daily_limit_pct": 0.08,
            "ann_vol": 0.28,
            "adv_contracts": 95_000,
        },
        {
            "contract": "GOLD",
            "asset_class": "metals",
            "initial_price": 2000.0,
            "tick_size": 0.10,
            "contract_multiplier": 100.0,
            "initial_margin_rate": 0.075,
            "maintenance_margin_rate": 0.065,
            "daily_limit_pct": 0.06,
            "ann_vol": 0.20,
            "adv_contracts": 180_000,
        },
        {
            "contract": "FX_CNH",
            "asset_class": "fx",
            "initial_price": 7.20,
            "tick_size": 0.0001,
            "contract_multiplier": 100_000.0,
            "initial_margin_rate": 0.055,
            "maintenance_margin_rate": 0.045,
            "daily_limit_pct": 0.04,
            "ann_vol": 0.10,
            "adv_contracts": 80_000,
        },
    ])

    return metadata

metadata = make_contract_metadata()
contracts = metadata["contract"].tolist()

metadata

## 6. Simulate futures prices with limit-board states

We simulate daily futures returns with:

- correlated factors;
- volatility regimes;
- stress periods;
- occasional limit-up/down events.

When a limit event occurs, the daily return is clipped near the limit band and trading is restricted in that direction.

In [ ]:
def round_price_to_tick(price, tick):
    return np.round(price / tick) * tick

def simulate_futures_prices(metadata, config):
    rng = np.random.default_rng(config.seed)
    dates = pd.bdate_range("2020-01-01", periods=config.n_days)

    contracts = metadata["contract"].tolist()
    n = len(contracts)

    corr = np.array([
        [1.00, -0.25, 0.35, 0.30, 0.25, 0.10],
        [-0.25, 1.00, -0.10, -0.10, 0.05, -0.15],
        [0.35, -0.10, 1.00, 0.50, 0.30, 0.10],
        [0.30, -0.10, 0.50, 1.00, 0.35, 0.10],
        [0.25, 0.05, 0.30, 0.35, 1.00, 0.05],
        [0.10, -0.15, 0.10, 0.10, 0.05, 1.00],
    ])

    ann_vol = metadata["ann_vol"].to_numpy()
    daily_cov = np.outer(ann_vol / np.sqrt(config.annualisation), ann_vol / np.sqrt(config.annualisation)) * corr

    prices = pd.DataFrame(index=dates, columns=contracts, dtype=float)
    returns = pd.DataFrame(index=dates, columns=contracts, dtype=float)
    limit_state = pd.DataFrame("normal", index=dates, columns=contracts)
    regime = pd.Series(0, index=dates, dtype=int)

    prices.iloc[0] = metadata["initial_price"].to_numpy()
    returns.iloc[0] = 0.0

    for t in range(1, config.n_days):
        if t > 1:
            if regime.iloc[t - 1] == 0:
                regime.iloc[t] = rng.choice([0, 1], p=[0.985, 0.015])
            else:
                regime.iloc[t] = rng.choice([0, 1], p=[0.06, 0.94])

        vol_mult = 1.0 if regime.iloc[t] == 0 else 2.3
        ret = rng.multivariate_normal(np.zeros(n), daily_cov * vol_mult**2)

        event_prob = config.limit_event_probability if regime.iloc[t] == 0 else config.stress_limit_event_probability

        for j, contract in enumerate(contracts):
            prev_price = prices.iloc[t - 1, j]
            limit_pct = metadata.loc[j, "daily_limit_pct"]

            if rng.uniform() < event_prob:
                direction = rng.choice([-1, 1])
                if direction > 0:
                    ret[j] = limit_pct * rng.uniform(0.96, 1.00)
                    limit_state.iloc[t, j] = "limit_up"
                else:
                    ret[j] = -limit_pct * rng.uniform(0.96, 1.00)
                    limit_state.iloc[t, j] = "limit_down"
            else:
                ret[j] = np.clip(ret[j], -limit_pct * 0.98, limit_pct * 0.98)
                limit_state.iloc[t, j] = "normal"

            new_price = prev_price * (1.0 + ret[j])
            tick = metadata.loc[j, "tick_size"]
            prices.iloc[t, j] = round_price_to_tick(new_price, tick)
            returns.iloc[t, j] = prices.iloc[t, j] / prev_price - 1.0

    regime_info = pd.DataFrame({"regime": regime}, index=dates)

    return prices, returns, limit_state, regime_info

prices, returns, limit_state, regime_info = simulate_futures_prices(metadata, config)

prices.head(), limit_state.stack().value_counts()

In [ ]:
plt.figure(figsize=(12, 6))
for contract in contracts:
    plt.plot(prices.index, prices[contract] / prices[contract].iloc[0], label=contract)
plt.title("Synthetic Futures Prices")
plt.xlabel("Date")
plt.ylabel("Normalised price")
plt.legend(ncol=3)
plt.show()

limit_counts = limit_state.stack().value_counts()
limit_counts

## 7. Build target weights

We use a simple cross-asset signal:

- 63-day momentum;
- 10-day reversal;
- 63-day volatility penalty.

The notebook is not about alpha discovery. The target signal simply creates realistic rebalancing pressure.

In [ ]:
def cross_sectional_zscore(df):
    mu = df.mean(axis=1)
    sd = df.std(axis=1).replace(0.0, np.nan)
    return df.sub(mu, axis=0).div(sd, axis=0).fillna(0.0)

def build_target_weights(prices, returns, config):
    momentum = cross_sectional_zscore(prices.pct_change(63))
    reversal = cross_sectional_zscore(-prices.pct_change(10))
    vol = returns.rolling(63).std() * np.sqrt(config.annualisation)
    low_vol = cross_sectional_zscore(-vol)

    signal = 0.55 * momentum + 0.25 * reversal + 0.20 * low_vol
    signal = signal.clip(-3, 3).fillna(0.0)

    inv_vol = 1.0 / vol.shift(1).replace(0.0, np.nan)
    inv_vol = inv_vol.fillna(inv_vol.expanding().median()).fillna(1.0)

    raw = signal.sub(signal.mean(axis=1), axis=0).mul(inv_vol, axis=0)
    weights = raw.div(raw.abs().sum(axis=1).replace(0.0, np.nan), axis=0) * config.gross_target
    weights = weights.fillna(0.0).clip(-config.max_abs_weight, config.max_abs_weight)

    gross = weights.abs().sum(axis=1).replace(0.0, np.nan)
    weights = weights.div(gross, axis=0).mul(config.gross_target).fillna(0.0)

    return signal, weights

signal, target_weights = build_target_weights(prices, returns, config)

target_weights.tail()

## 8. Contract conversion functions

Convert target weights to contracts.

For futures:

$$
notional_i = price_i \times multiplier_i \times contracts_i
$$

Target contracts:

$$
contracts_i^* = \frac{targetWeight_i \times equity} {price_i \times multiplier_i}
$$

Contracts must be rounded to integers.

In [ ]:
def contract_notional(price_row, metadata):
    multipliers = metadata.set_index("contract")["contract_multiplier"].reindex(price_row.index)
    return price_row * multipliers

def target_weights_to_contracts(target_weight_row, price_row, equity, metadata):
    notionals = contract_notional(price_row, metadata)
    target_contracts = target_weight_row * equity / notionals
    return target_contracts

def round_contracts(contract_series):
    return np.round(contract_series).astype(int)

def contracts_to_weights(contracts_row, price_row, equity, metadata):
    notionals = contract_notional(price_row, metadata)
    return contracts_row * notionals / equity

example_target_contracts = target_weights_to_contracts(
    target_weights.iloc[-1],
    prices.iloc[-1],
    config.initial_equity,
    metadata,
)

example_target_contracts, round_contracts(example_target_contracts)

## 9. Margin model

Initial margin:

$$
IM_i = |contracts_i| \times price_i \times multiplier_i \times initialMarginRate_i
$$

Maintenance margin:

$$
MM_i = |contracts_i| \times price_i \times multiplier_i \times maintenanceMarginRate_i
$$

Margin utilisation:

$$
utilisation = \frac{\sum_i IM_i}{equity}
$$

If utilisation exceeds a threshold, the portfolio must deleverage.

In [ ]:
def margin_requirements(contracts_row, price_row, metadata):
    meta = metadata.set_index("contract").reindex(contracts_row.index)
    notional_abs = contracts_row.abs() * price_row * meta["contract_multiplier"]
    initial_margin = notional_abs * meta["initial_margin_rate"]
    maintenance_margin = notional_abs * meta["maintenance_margin_rate"]

    return pd.DataFrame({
        "contract": contracts_row.index,
        "contracts": contracts_row.to_numpy(),
        "abs_notional": notional_abs.to_numpy(),
        "initial_margin": initial_margin.to_numpy(),
        "maintenance_margin": maintenance_margin.to_numpy(),
    })

def total_margin(contracts_row, price_row, metadata):
    detail = margin_requirements(contracts_row, price_row, metadata)
    return {
        "initial_margin": detail["initial_margin"].sum(),
        "maintenance_margin": detail["maintenance_margin"].sum(),
        "detail": detail,
    }

zero_contracts = pd.Series(0, index=contracts)
total_margin(zero_contracts, prices.iloc[-1], metadata)["initial_margin"]

## 10. Deadband policies

We implement several policies:

### None

Always rebalance to rounded target.

### Fixed deadband

Trade only if absolute weight drift exceeds a fixed band.

### Volatility-scaled deadband

Wider band for higher-volatility contracts:

$$
band_i = baseBand \times \frac{\sigma_i}{median(\sigma)}
$$

This reduces churn in naturally noisy contracts.

In [ ]:
def get_deadband(policy_name, date_idx, returns, metadata, config):
    if policy_name == "none":
        return pd.Series(0.0, index=contracts)

    if policy_name == "small":
        return pd.Series(config.base_deadband, index=contracts)

    if policy_name == "medium":
        return pd.Series(config.medium_deadband, index=contracts)

    if policy_name == "large":
        return pd.Series(config.large_deadband, index=contracts)

    if policy_name == "vol_scaled":
        vol = returns.iloc[:date_idx + 1].tail(63).std() * np.sqrt(config.annualisation)
        if vol.isna().all() or vol.median() == 0 or pd.isna(vol.median()):
            return pd.Series(config.medium_deadband, index=contracts)
        scaled = config.base_deadband * vol / vol.median()
        return scaled.clip(config.base_deadband * 0.5, config.large_deadband)

    raise ValueError(f"Unknown deadband policy: {policy_name}")

for policy in ["none", "small", "medium", "large", "vol_scaled"]:
    print(policy, get_deadband(policy, 100, returns, metadata, config).round(4).to_dict())

## 11. Trade blocking from limit boards

Simplified rules:

| Limit state | Buy trade | Sell trade |
|---|---|---|
| normal | allowed | allowed |
| limit up | blocked | allowed |
| limit down | allowed | blocked |

A blocked trade creates tracking error and missed rebalance attribution.

In [ ]:
def is_trade_blocked(trade_contracts, limit_state_row):
    blocked = pd.Series(False, index=trade_contracts.index)

    for contract in trade_contracts.index:
        trade = trade_contracts[contract]
        state = limit_state_row[contract]

        if trade > 0 and state == "limit_up":
            blocked[contract] = True
        elif trade < 0 and state == "limit_down":
            blocked[contract] = True

    return blocked

## 12. Rebalancing engine

The engine applies:

1. target weight calculation;
2. contract rounding;
3. deadband check;
4. price-limit blocking;
5. margin check;
6. forced deleveraging;
7. PnL and variation margin;
8. cost accounting.

This is the core of the notebook.

In [ ]:
def run_rebalance_engine(policy_name, prices, returns, target_weights, limit_state, metadata, config):
    dates = prices.index
    contracts_list = metadata["contract"].tolist()

    current_contracts = pd.Series(0, index=contracts_list, dtype=int)
    equity = config.initial_equity
    cash = config.initial_equity

    rows = []
    trade_rows = []
    margin_rows = []

    prev_price = prices.iloc[0]

    for t, date in enumerate(dates):
        price = prices.iloc[t]

        # Variation margin PnL from previous close to current close.
        if t > 0:
            price_change = price - prev_price
            multiplier = metadata.set_index("contract")["contract_multiplier"].reindex(contracts_list)
            pnl_by_contract = current_contracts * price_change * multiplier
            daily_pnl = pnl_by_contract.sum()
            equity += daily_pnl
            cash += daily_pnl
        else:
            daily_pnl = 0.0
            pnl_by_contract = pd.Series(0.0, index=contracts_list)

        current_weights = contracts_to_weights(current_contracts, price, equity, metadata)
        desired_weights = target_weights.iloc[t].reindex(contracts_list).fillna(0.0)

        target_contracts_float = target_weights_to_contracts(desired_weights, price, equity, metadata)
        target_contracts = round_contracts(target_contracts_float)

        desired_trade = target_contracts - current_contracts

        deadband = get_deadband(policy_name, t, returns, metadata, config)
        weight_drift = desired_weights - current_weights
        trade_allowed_by_deadband = weight_drift.abs() > deadband

        trade_after_deadband = desired_trade.where(trade_allowed_by_deadband, 0)

        blocked = is_trade_blocked(trade_after_deadband, limit_state.iloc[t])
        executable_trade = trade_after_deadband.where(~blocked, 0)

        # Trading cost: commission plus slippage ticks.
        multiplier = metadata.set_index("contract")["contract_multiplier"].reindex(contracts_list)
        tick_size = metadata.set_index("contract")["tick_size"].reindex(contracts_list)

        commission_cost = executable_trade.abs() * config.commission_per_contract
        slippage_cost = executable_trade.abs() * config.slippage_ticks * tick_size * multiplier
        total_trade_cost_by_contract = commission_cost + slippage_cost
        total_trade_cost = total_trade_cost_by_contract.sum()

        proposed_contracts = current_contracts + executable_trade

        margin = total_margin(proposed_contracts, price, metadata)
        margin_util = margin["initial_margin"] / max(equity, 1.0)

        forced_deleverage = False
        deleverage_trade = pd.Series(0, index=contracts_list, dtype=int)

        # If margin too high, reduce all positions proportionally.
        if margin_util > config.max_margin_utilisation:
            forced_deleverage = True
            reduced_contracts = np.round(proposed_contracts * (1.0 - config.forced_deleverage_fraction)).astype(int)
            deleverage_trade = reduced_contracts - proposed_contracts
            proposed_contracts = reduced_contracts

            extra_commission = deleverage_trade.abs() * config.commission_per_contract
            extra_slippage = deleverage_trade.abs() * config.slippage_ticks * tick_size * multiplier
            total_trade_cost += (extra_commission + extra_slippage).sum()

            margin = total_margin(proposed_contracts, price, metadata)
            margin_util = margin["initial_margin"] / max(equity, 1.0)

        equity -= total_trade_cost
        cash -= total_trade_cost

        current_contracts = proposed_contracts.astype(int)

        actual_weights = contracts_to_weights(current_contracts, price, equity, metadata)
        tracking_error_weight = actual_weights - desired_weights

        rows.append({
            "date": date,
            "policy": policy_name,
            "equity": equity,
            "cash": cash,
            "daily_pnl": daily_pnl,
            "trade_cost": total_trade_cost,
            "gross_target_weight": desired_weights.abs().sum(),
            "gross_actual_weight": actual_weights.abs().sum(),
            "net_actual_weight": actual_weights.sum(),
            "gross_weight_tracking_error": tracking_error_weight.abs().sum(),
            "max_abs_weight_tracking_error": tracking_error_weight.abs().max(),
            "turnover_contracts": executable_trade.abs().sum(),
            "turnover_notional": (executable_trade.abs() * price * multiplier).sum(),
            "blocked_contract_count": int(blocked.sum()),
            "blocked_trade_notional": (trade_after_deadband.where(blocked, 0).abs() * price * multiplier).sum(),
            "deadband_suppressed_contract_count": int(((desired_trade != 0) & (trade_after_deadband == 0)).sum()),
            "forced_deleverage": forced_deleverage,
            "initial_margin": margin["initial_margin"],
            "maintenance_margin": margin["maintenance_margin"],
            "margin_utilisation": margin_util,
        })

        for contract in contracts_list:
            trade_rows.append({
                "date": date,
                "policy": policy_name,
                "contract": contract,
                "desired_contracts": int(target_contracts[contract]),
                "current_contracts_after": int(current_contracts[contract]),
                "desired_trade": int(desired_trade[contract]),
                "trade_after_deadband": int(trade_after_deadband[contract]),
                "blocked_by_limit": bool(blocked[contract]),
                "executed_trade": int(executable_trade[contract]),
                "deleverage_trade": int(deleverage_trade[contract]),
                "limit_state": limit_state.iloc[t][contract],
                "deadband": float(deadband[contract]),
                "weight_drift": float(weight_drift[contract]),
                "actual_weight": float(actual_weights[contract]),
                "target_weight": float(desired_weights[contract]),
            })

        margin_detail = margin["detail"].copy()
        margin_detail["date"] = date
        margin_detail["policy"] = policy_name
        margin_rows.append(margin_detail)

        prev_price = price.copy()

    portfolio = pd.DataFrame(rows).set_index("date")
    trades = pd.DataFrame(trade_rows)
    margin_detail = pd.concat(margin_rows, ignore_index=True)

    portfolio["daily_return"] = portfolio["equity"].pct_change().fillna(0.0)
    portfolio["nav"] = portfolio["equity"] / config.initial_equity

    return portfolio, trades, margin_detail

policies = ["none", "small", "medium", "large", "vol_scaled"]

engine_outputs = {}
for policy in policies:
    engine_outputs[policy] = run_rebalance_engine(policy, prices, returns, target_weights, limit_state, metadata, config)

portfolio_none, trades_none, margin_none = engine_outputs["none"]

portfolio_none.head()

## 13. Portfolio performance and constraint metrics

In [ ]:
def max_drawdown(nav):
    dd = nav / nav.cummax() - 1.0
    return dd, float(dd.min())

def historical_var_cvar(losses, alpha=0.95):
    losses = pd.Series(losses).dropna()
    var = losses.quantile(alpha)
    tail = losses[losses >= var]
    return float(var), float(tail.mean()) if len(tail) else np.nan

def summarise_policy(portfolio, policy, config):
    r = portfolio["daily_return"].dropna()
    nav = portfolio["nav"].dropna()
    _, mdd = max_drawdown(nav)
    var, cvar = historical_var_cvar(-r)

    ann_return = (1 + r).prod() ** (config.annualisation / len(r)) - 1
    ann_vol = r.std(ddof=1) * np.sqrt(config.annualisation)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan

    return {
        "policy": policy,
        "annualised_return": ann_return,
        "annualised_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": mdd,
        "VaR95": var,
        "CVaR95": cvar,
        "total_return": nav.iloc[-1] - 1,
        "annualised_trade_cost": portfolio["trade_cost"].mean() * config.annualisation / config.initial_equity,
        "mean_margin_utilisation": portfolio["margin_utilisation"].mean(),
        "p95_margin_utilisation": portfolio["margin_utilisation"].quantile(0.95),
        "max_margin_utilisation": portfolio["margin_utilisation"].max(),
        "mean_gross_tracking_error": portfolio["gross_weight_tracking_error"].mean(),
        "p95_gross_tracking_error": portfolio["gross_weight_tracking_error"].quantile(0.95),
        "total_turnover_notional": portfolio["turnover_notional"].sum(),
        "total_trade_cost": portfolio["trade_cost"].sum(),
        "blocked_trade_days": int((portfolio["blocked_contract_count"] > 0).sum()),
        "forced_deleverage_days": int(portfolio["forced_deleverage"].sum()),
        "deadband_suppressed_days": int((portfolio["deadband_suppressed_contract_count"] > 0).sum()),
    }

policy_summaries = []
for policy in policies:
    portfolio, trades, margin_detail = engine_outputs[policy]
    policy_summaries.append(summarise_policy(portfolio, policy, config))

policy_summary = pd.DataFrame(policy_summaries).sort_values("sharpe", ascending=False)

policy_summary

In [ ]:
plt.figure(figsize=(12, 6))
for policy in policies:
    portfolio = engine_outputs[policy][0]
    plt.plot(portfolio.index, portfolio["nav"], label=policy)
plt.title("NAV by Deadband Policy")
plt.xlabel("Date")
plt.ylabel("NAV")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.barh(policy_summary["policy"], policy_summary["total_trade_cost"])
plt.title("Total Trading Cost by Policy")
plt.xlabel("Trading cost")
plt.ylabel("Policy")
plt.show()

## 14. Turnover versus tracking-error trade-off

Deadbands reduce turnover, but increase weight tracking error.

This is the central trade-off.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(
    policy_summary["total_turnover_notional"],
    policy_summary["mean_gross_tracking_error"],
    s=100,
)
for _, row in policy_summary.iterrows():
    plt.text(row["total_turnover_notional"], row["mean_gross_tracking_error"], row["policy"])
plt.title("Turnover vs Tracking Error")
plt.xlabel("Total turnover notional")
plt.ylabel("Mean gross weight tracking error")
plt.grid(True, alpha=0.3)
plt.show()

## 15. Margin utilisation diagnostics

Margin pressure can force deleveraging and distort strategy behaviour.

We inspect margin utilisation by policy.

In [ ]:
plt.figure(figsize=(12, 6))
for policy in policies:
    portfolio = engine_outputs[policy][0]
    plt.plot(portfolio.index, portfolio["margin_utilisation"], label=policy, alpha=0.8)
plt.axhline(config.max_margin_utilisation, linestyle="--", label="max margin utilisation")
plt.title("Margin Utilisation by Policy")
plt.xlabel("Date")
plt.ylabel("Initial margin / equity")
plt.legend()
plt.show()

## 16. Limit-board missed-trade attribution

When a target trade is blocked by limit-up/down, the portfolio cannot reach target.

We attribute blocked trades by:

- contract;
- limit state;
- policy;
- notional value.

In [ ]:
blocked_reports = []

for policy in policies:
    trades = engine_outputs[policy][1]
    portfolio = engine_outputs[policy][0]

    blocked = trades[trades["blocked_by_limit"]].copy()
    if len(blocked) == 0:
        continue

    # Estimate blocked notional using same-day price and multiplier.
    price_stack = prices.stack().rename("price").reset_index().rename(columns={"level_0": "date", "level_1": "contract"})
    meta = metadata[["contract", "contract_multiplier"]]
    blocked = blocked.merge(price_stack, on=["date", "contract"], how="left").merge(meta, on="contract", how="left")
    blocked["blocked_notional"] = blocked["trade_after_deadband"].abs() * blocked["price"] * blocked["contract_multiplier"]

    report = (
        blocked
        .groupby(["policy", "contract", "limit_state"])
        .agg(
            n_blocked=("blocked_by_limit", "count"),
            total_blocked_notional=("blocked_notional", "sum"),
            mean_abs_weight_drift=("weight_drift", lambda x: x.abs().mean()),
        )
        .reset_index()
    )
    blocked_reports.append(report)

blocked_trade_report = pd.concat(blocked_reports, ignore_index=True) if blocked_reports else pd.DataFrame()

blocked_trade_report.head(20)

In [ ]:
if len(blocked_trade_report):
    plot_df = (
        blocked_trade_report
        .groupby("contract")["total_blocked_notional"]
        .sum()
        .sort_values()
    )

    plt.figure(figsize=(10, 5))
    plt.barh(plot_df.index, plot_df.values)
    plt.title("Blocked Trade Notional by Contract")
    plt.xlabel("Blocked notional")
    plt.ylabel("Contract")
    plt.show()

## 17. Forced deleveraging diagnostics

When margin utilisation breaches the threshold, the engine reduces positions.

This is simplified but captures the point:

> margin can force trades that have nothing to do with alpha.

In [ ]:
forced_rows = []

for policy in policies:
    portfolio = engine_outputs[policy][0]
    trades = engine_outputs[policy][1]

    forced_days = portfolio[portfolio["forced_deleverage"]].index

    forced_rows.append({
        "policy": policy,
        "forced_deleverage_days": len(forced_days),
        "first_forced_date": forced_days.min() if len(forced_days) else pd.NaT,
        "max_margin_utilisation": portfolio["margin_utilisation"].max(),
        "mean_margin_utilisation": portfolio["margin_utilisation"].mean(),
    })

forced_deleverage_report = pd.DataFrame(forced_rows)

forced_deleverage_report

## 18. Contract-level realised holdings

Inspect actual contract holdings through time.

Deadbands and limit boards create path dependence.

In [ ]:
def contract_holdings_table(trades):
    holdings = trades.pivot(index="date", columns="contract", values="current_contracts_after").sort_index()
    return holdings

holdings_by_policy = {policy: contract_holdings_table(engine_outputs[policy][1]) for policy in policies}

holdings_by_policy["medium"].tail()

In [ ]:
plt.figure(figsize=(12, 6))
medium_holdings = holdings_by_policy["medium"]
for contract in contracts:
    plt.plot(medium_holdings.index, medium_holdings[contract], label=contract)
plt.title("Actual Contract Holdings — Medium Deadband")
plt.xlabel("Date")
plt.ylabel("Contracts")
plt.legend(ncol=3)
plt.show()

## 19. Deadband event examples

Find days where the deadband suppressed many trades.

These are useful audit examples.

In [ ]:
medium_portfolio, medium_trades, medium_margin = engine_outputs["medium"]

deadband_examples = medium_portfolio.sort_values("deadband_suppressed_contract_count", ascending=False).head(10)[
    [
        "equity",
        "gross_actual_weight",
        "gross_weight_tracking_error",
        "deadband_suppressed_contract_count",
        "turnover_notional",
        "trade_cost",
    ]
]

deadband_examples

## 20. Stress period analysis

Compare normal and stress regimes.

In [ ]:
stress_reports = []

for policy in policies:
    portfolio = engine_outputs[policy][0].join(regime_info, how="left")

    report = (
        portfolio
        .groupby("regime")
        .agg(
            n_days=("daily_return", "count"),
            mean_daily_return=("daily_return", "mean"),
            daily_vol=("daily_return", "std"),
            mean_margin_utilisation=("margin_utilisation", "mean"),
            mean_tracking_error=("gross_weight_tracking_error", "mean"),
            mean_trade_cost=("trade_cost", "mean"),
            blocked_days=("blocked_contract_count", lambda x: (x > 0).sum()),
            forced_deleverage_days=("forced_deleverage", "sum"),
        )
        .reset_index()
    )

    report["policy"] = policy
    report["ann_return_proxy"] = report["mean_daily_return"] * config.annualisation
    report["ann_vol_proxy"] = report["daily_vol"] * np.sqrt(config.annualisation)
    stress_reports.append(report)

stress_report = pd.concat(stress_reports, ignore_index=True)

stress_report

## 21. Price-limit board dashboard

Summarise how often each contract hit limit up/down and how much it affected trading.

In [ ]:
limit_dashboard_rows = []

for contract in contracts:
    states = limit_state[contract]
    limit_dashboard_rows.append({
        "contract": contract,
        "limit_up_days": int((states == "limit_up").sum()),
        "limit_down_days": int((states == "limit_down").sum()),
        "any_limit_days": int((states != "normal").sum()),
    })

limit_dashboard = pd.DataFrame(limit_dashboard_rows)

if len(blocked_trade_report):
    blocked_by_contract = (
        blocked_trade_report
        .groupby("contract")
        .agg(
            total_blocked_notional=("total_blocked_notional", "sum"),
            total_blocked_events=("n_blocked", "sum"),
        )
        .reset_index()
    )
    limit_dashboard = limit_dashboard.merge(blocked_by_contract, on="contract", how="left")
else:
    limit_dashboard["total_blocked_notional"] = 0.0
    limit_dashboard["total_blocked_events"] = 0

limit_dashboard = limit_dashboard.fillna(0).sort_values("any_limit_days", ascending=False)

limit_dashboard

## 22. Governance flags

A futures constraint engine should be reviewed if:

1. margin utilisation breaches threshold too often;
2. forced deleveraging occurs;
3. limit boards block meaningful notional;
4. deadband tracking error is high;
5. turnover remains high despite deadbands;
6. contract rounding creates excessive tracking error;
7. stress regime behaviour deteriorates.

In [ ]:
selected_policy = "medium"
selected_summary = policy_summary[policy_summary["policy"] == selected_policy].iloc[0]
selected_portfolio = engine_outputs[selected_policy][0]

blocked_notional_selected = 0.0
if len(blocked_trade_report):
    blocked_notional_selected = blocked_trade_report.loc[
        blocked_trade_report["policy"] == selected_policy,
        "total_blocked_notional",
    ].sum()

stress_selected = stress_report[(stress_report["policy"] == selected_policy) & (stress_report["regime"] == 1)]
stress_tracking_error = float(stress_selected["mean_tracking_error"].iloc[0]) if len(stress_selected) else np.nan

governance_flags = pd.DataFrame([{
    "selected_policy": selected_policy,
    "sharpe": selected_summary["sharpe"],
    "max_drawdown": selected_summary["max_drawdown"],
    "annualised_trade_cost": selected_summary["annualised_trade_cost"],
    "mean_margin_utilisation": selected_summary["mean_margin_utilisation"],
    "p95_margin_utilisation": selected_summary["p95_margin_utilisation"],
    "max_margin_utilisation": selected_summary["max_margin_utilisation"],
    "forced_deleverage_days": selected_summary["forced_deleverage_days"],
    "blocked_trade_days": selected_summary["blocked_trade_days"],
    "blocked_notional": blocked_notional_selected,
    "mean_tracking_error": selected_summary["mean_gross_tracking_error"],
    "p95_tracking_error": selected_summary["p95_gross_tracking_error"],
    "stress_tracking_error": stress_tracking_error,
    "flag_margin_utilisation_high": bool(selected_summary["p95_margin_utilisation"] > config.max_margin_utilisation),
    "flag_forced_deleveraging": bool(selected_summary["forced_deleverage_days"] > 0),
    "flag_blocked_notional_high": bool(blocked_notional_selected > config.initial_equity * 0.50),
    "flag_tracking_error_high": bool(selected_summary["p95_gross_tracking_error"] > 0.30),
    "flag_trade_cost_high": bool(selected_summary["annualised_trade_cost"] > 0.05),
    "flag_stress_tracking_error_high": bool(stress_tracking_error > 0.35) if np.isfinite(stress_tracking_error) else False,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_margin_utilisation_high",
        "flag_forced_deleveraging",
        "flag_blocked_notional_high",
        "flag_tracking_error_high",
        "flag_trade_cost_high",
        "flag_stress_tracking_error_high",
    ]
].any(axis=1)

governance_flags

## 23. Audit checks

Numerical and process checks:

1. prices are positive;
2. prices lie on tick grid;
3. target weights are finite;
4. contract holdings are integer;
5. equity stays positive;
6. margin is non-negative;
7. limit-state labels are valid;
8. governance flags exist.

In [ ]:
audit_rows = []

prices_positive = bool((prices > 0).all().all())
audit_rows.append({
    "check": "prices_positive",
    "value": prices_positive,
    "passed": prices_positive,
})

tick_ok = True
for _, row in metadata.iterrows():
    contract = row["contract"]
    tick = row["tick_size"]
    values = prices[contract].to_numpy()
    tick_ok = tick_ok and np.allclose(values / tick, np.round(values / tick), atol=1e-8)

audit_rows.append({
    "check": "prices_on_tick_grid",
    "value": tick_ok,
    "passed": bool(tick_ok),
})

weights_finite = bool(np.isfinite(target_weights.to_numpy()).all())
audit_rows.append({
    "check": "target_weights_finite",
    "value": weights_finite,
    "passed": weights_finite,
})

holdings_integer = True
for policy, holdings in holdings_by_policy.items():
    holdings_integer = holdings_integer and np.allclose(holdings.to_numpy(), np.round(holdings.to_numpy()))

audit_rows.append({
    "check": "contract_holdings_integer",
    "value": holdings_integer,
    "passed": bool(holdings_integer),
})

equity_positive = bool(all((engine_outputs[p][0]["equity"] > 0).all() for p in policies))
audit_rows.append({
    "check": "equity_positive_all_policies",
    "value": equity_positive,
    "passed": equity_positive,
})

margin_nonnegative = bool(all((engine_outputs[p][0]["initial_margin"] >= 0).all() for p in policies))
audit_rows.append({
    "check": "margin_nonnegative_all_policies",
    "value": margin_nonnegative,
    "passed": margin_nonnegative,
})

valid_states = {"normal", "limit_up", "limit_down"}
limit_states_valid = bool(set(limit_state.stack().unique()).issubset(valid_states))
audit_rows.append({
    "check": "limit_states_valid",
    "value": limit_states_valid,
    "passed": limit_states_valid,
})

governance_exists = bool(len(governance_flags) == 1)
audit_rows.append({
    "check": "governance_flags_exist",
    "value": governance_exists,
    "passed": governance_exists,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 24. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed") / config.output_subdir
output_dir.mkdir(parents=True, exist_ok=True)

metadata.to_csv(output_dir / "contract_metadata.csv", index=False)
prices.to_csv(output_dir / "futures_prices.csv")
returns.to_csv(output_dir / "futures_returns.csv")
limit_state.to_csv(output_dir / "limit_state.csv")
regime_info.to_csv(output_dir / "regime_info.csv")
signal.to_csv(output_dir / "signal.csv")
target_weights.to_csv(output_dir / "target_weights.csv")
policy_summary.to_csv(output_dir / "policy_summary.csv", index=False)
blocked_trade_report.to_csv(output_dir / "blocked_trade_report.csv", index=False)
forced_deleverage_report.to_csv(output_dir / "forced_deleverage_report.csv", index=False)
stress_report.to_csv(output_dir / "stress_report.csv", index=False)
limit_dashboard.to_csv(output_dir / "limit_dashboard.csv", index=False)
governance_flags.to_csv(output_dir / "governance_flags.csv", index=False)
audit_report.to_csv(output_dir / "audit_report.csv", index=False)

for policy in policies:
    portfolio, trades, margin_detail = engine_outputs[policy]
    portfolio.to_csv(output_dir / f"portfolio_{policy}.csv")
    trades.to_csv(output_dir / f"trades_{policy}.csv", index=False)
    margin_detail.to_csv(output_dir / f"margin_detail_{policy}.csv", index=False)
    holdings_by_policy[policy].to_csv(output_dir / f"holdings_{policy}.csv")

manifest = {
    "dataset_name": "limit_board_margin_deadband_rebalancing_outputs",
    "schema_version": "limit_board_margin_deadband_rebalancing_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "contracts": contracts,
    "policies": policies,
    "selected_policy": selected_policy,
    "policy_summary": policy_summary.to_dict(orient="records"),
    "limit_dashboard": limit_dashboard.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "This notebook models futures-specific implementation constraints.",
        "Constraints include contract rounding, daily price-limit boards, margin utilisation and deadband rebalancing.",
        "Limit-up blocks buy trades and limit-down blocks sell trades in the simplified execution model.",
        "Deadbands reduce turnover at the cost of tracking error.",
        "Forced deleveraging is triggered when margin utilisation exceeds the configured threshold.",
        "The notebook is educational and uses synthetic futures data."
    ],
}

manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

output_dir / "policy_summary.csv", output_dir / "limit_dashboard.csv", output_dir / "governance_flags.csv", manifest_path

## 25. Practical checklist for futures rebalancing systems

Before trusting a futures portfolio engine:

1. **Store contract metadata explicitly.**
2. **Round prices to ticks.**
3. **Round positions to integer contracts.**
4. **Model contract multipliers.**
5. **Apply initial and maintenance margin.**
6. **Track margin utilisation daily.**
7. **Model price-limit boards.**
8. **Attribute blocked trades.**
9. **Use deadbands to control churn.**
10. **Measure tracking error caused by deadbands and blocked trades.**

## 26. Limitations

### Simplified limit-board rules

Real exchange rules are more subtle. A limit-up market does not always mean every buy is impossible, and order priority matters.

### Daily frequency

The engine is daily. Intraday limit events and margin calls can be more complex.

### Synthetic futures data

Real futures need exchange calendars, contract rolls, settlement prices, night sessions, holiday schedules, and actual limit rules.

### Simplified margin

Margin is percentage-based. Real margin is exchange/broker-specific and can change through time.

### Simplified deleveraging

Forced deleveraging is proportional. A production engine should optimise which contracts to reduce.

### No slippage from liquidity depth

Slippage is tick-based and simple. It does not use full order book depth.

## 27. Research frontier and extensions

Important extensions include:

1. exchange-specific limit-board mechanics;
2. settlement-price-based margin;
3. intraday margin and risk checks;
4. futures contract rolling;
5. night-session liquidity;
6. price-limit queue modelling;
7. margin-aware optimisation;
8. deadband optimisation;
9. forced deleveraging optimisation;
10. China futures exchange rule modelling with tick size, lot size, margin and price-limit tables.

## 28. Suggested follow-up notebooks

This notebook naturally leads to:

1. **06_08_microprice_and_short_horizon_alpha**  
   Use L1 bid/ask information for short-horizon execution signals.

2. **06_09_execution_risk_controls_and_kill_switch**  
   Add hard controls for limit boards and margin stress.

3. **06_10_futures_roll_execution_engine**  
   Extend contract metadata to rolling and expiry.

4. **06_11_l1_bid_ask_backtest_execution_model**  
   Use L1 fills with futures-style constraints.

5. **06_12_production_reconciliation_and_breaks**  
   Reconcile target, executable and actual futures positions.

## 29. Summary

This notebook implemented limit-board, margin and deadband rebalancing.

It showed how to:

1. define futures contract metadata;
2. simulate tick-rounded futures prices;
3. simulate daily limit-up and limit-down states;
4. generate target weights;
5. convert weights to integer contracts;
6. compute initial and maintenance margin;
7. apply deadband rebalancing policies;
8. block trades during price-limit boards;
9. account for trade costs;
10. trigger forced deleveraging under margin pressure;
11. compare deadband policies;
12. quantify turnover versus tracking-error trade-offs;
13. attribute blocked trades by contract;
14. monitor margin utilisation;
15. analyse stress regimes;
16. build governance flags and audit checks;
17. save outputs and manifests.

The key computational takeaway:

> A futures rebalancing engine must translate continuous target weights into discrete, margin-aware, limit-aware contract trades.

The key financial takeaway:

> The best target portfolio is irrelevant if the exchange, margin account, or deadband policy prevents you from reaching it.

## 30. Further reading

- Hull, *Options, Futures, and Other Derivatives.*
- CME and futures exchange rulebooks on daily price limits and margin.
- Harris, *Trading and Exchanges.*
- Cartea, Jaimungal and Penalva, *Algorithmic and High-Frequency Trading.*
- Kissell, *The Science of Algorithmic Trading and Portfolio Management.*
- Exchange documentation for tick sizes, contract multipliers, margin schedules and price-limit rules.